<a href="https://colab.research.google.com/github/muskanrai-20/Shortify/blob/main/Shortify1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [4]:
from transformers import T5Tokenizer , Trainer ,TrainingArguments,T5ForConditionalGeneration

In [5]:
from google.colab import files

# This will open a file selection dialog in your browser
uploaded = files.upload()

Saving samsum-train.csv to samsum-train.csv


In [6]:
from google.colab import files

# This will open a file selection dialog in your browser
uploaded = files.upload()

Saving samsum-validation.csv to samsum-validation.csv


In [7]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [9]:
# Random Sample

train_data = train_data.sample(n=4000, random_state= 42).reset_index(drop=True)

val_data = val_data.sample(n=500, random_state= 42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

## Data preprocessing

In [11]:
import re


def clean_data(text):
  text = re.sub(r"/r/n"," ",text)
  text = re.sub(r"/s+/"," ",text)
  text = re.sub(r"<.*?>"," ",text)


  text = text.strip().lower()
  return text


In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)



## Tokenization

In [13]:



tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [14]:
# raw dat >>> tokenized inputs (for fine tuning)


def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding ="max_length", max_length = 512, truncation = True)

  targets = tokenizer(data["summary"], padding ="max_length", max_length = 512, truncation = True)

  inputs["labels"] = targets["input_ids"]
  return inputs

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()

val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [16]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
len(train_dataset[0]["input_ids"])

512

In [18]:
type(train_dataset)

type(val_dataset)

list

## WORK WITH MODEL

In [19]:
# NLP >> Generationtask

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [20]:
import torch


if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device =  torch.device("cpu")

  print("device : ",device)
  model.to(device)


In [21]:
# Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps=500,


)

In [22]:
import torch
import pandas as pd
import re
from transformers import Trainer, T5ForConditionalGeneration, T5Tokenizer

# Define device
if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device =  torch.device("cpu")

# Initialize model and move to device
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.to(device)

# Re-initialize tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# --- Data Loading and Preprocessing (re-added for self-containment) ---
# Load data
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

# Sample data
train_data = train_data.sample(n=4000, random_state= 42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state= 42).reset_index(drop=True)

# Define clean_data function
def clean_data(text):
  text = re.sub(r"/r/n"," ",text)
  text = re.sub(r"/s+/"," ",text)
  text = re.sub(r"<.*?>"," ",text)
  text = text.strip().lower()
  return text

# Apply clean_data
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)
# --- End Data Loading and Preprocessing ---


# Re-define tokenize function
def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding ="max_length", max_length = 512, truncation = True)
  targets = tokenizer(data["summary"], padding ="max_length", max_length = 512, truncation = True)
  inputs["labels"] = targets["input_ids"]
  return inputs

# Re-create train_dataset and val_dataset.
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

trainer = Trainer(
    model= model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.813140,0.114055
2,0.120010,0.107213
3,0.112369,0.105341
4,0.108913,0.103933
5,0.106758,0.103491
6,0.105757,0.103242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.7278243687947591, metrics={'train_runtime': 2299.7891, 'train_samples_per_second': 10.436, 'train_steps_per_second': 1.304, 'total_flos': 3248203235328000.0, 'train_loss': 0.7278243687947591, 'epoch': 6.0})

In [27]:
#model load >> fine tune >>save the model

model.save_pretrained("./Saved_summary_model")

tokenizer.save_pretrained("./Saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./Saved_summary_model/tokenizer_config.json',
 './Saved_summary_model/tokenizer.json')

In [29]:
model.from_pretrained("./Saved_summary_model")

tokenizer.from_pretrained("./Saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='./Saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False

## Test the core logic

In [34]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean

  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"

  ).to(device)

  targets = model.generate(
     input_ids = inputs["input_ids"],
     attention_mask = inputs["attention_mask"],
     max_length = 150,
     num_beams=4,
     early_stopping = True,

  )

  summary = tokenizer.decode(targets[0],skip_special_tokens = True)
  return summary

In [37]:
test_dialogue = """The Lost Letter

“Excuse me, is this yours?” Aarav asked, holding a folded paper.

Meera looked up. “Mine? I don’t think so.”

Aarav opened it carefully. “It says, To the person who finds this…”

Meera smiled. “Now I’m curious.”

He read aloud, “If you found this letter, tell someone something kind today.”

They both laughed.

“That’s unexpected,” Meera said.

Aarav thought for a second and smiled. “Okay then… your smile made my day better.”

Meera laughed softly. “And you actually followed the letter.”

She looked at him and said, “You seem like someone who makes strangers comfortable.”

Aarav folded the letter again. “Should we leave it somewhere for the next person?”

Meera nodded. “And maybe add a line.”

She wrote: Kindness travels faster than we think.

They left the letter on a park bench and walked away, wondering who would find it next.

The End"""

In [38]:
summary = summarize_dialogue(test_dialogue)


print("Summary : ", summary)

Summary :  aarav found the lost letter on a park bench and walked away wondering who would find it next. kindness travels faster than we think.
